In [4]:
import sys
print(sys.executable)

c:\Users\mubee\AppData\Local\Programs\Python\Python314\python.exe


In [5]:
import sys
!{sys.executable} -m pip install pandas sqlalchemy mysql-connector-python

  Using cached tzdata-2026.4-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/10.0 MB 2.7 MB/s eta 0:00:04
   ------ --------------------------------- 1.6/10.0 MB 2.5 MB/s eta 0:00:04
   -------------- ------------------------- 3.7/10.0 MB 5.0 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/10.0 MB 5.3 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/10.0 MB 5.7 MB/s eta 0:00:01
   --------------------------------- ------ 8.4/10.0 MB 5.9 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.0 MB 6.0 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 5.7 MB/s  0:00:02
   -----------------------------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install mysql-connector-python
import mysql.connector
import pandas as pd
import getpass
from sqlalchemy import create_engine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
# Database credentials
username = "root"
password = getpass.getpass("Enter MySQL password: ")  # password hidden
host = "localhost"
database = "pgdsms"

In [12]:
import mysql.connector

# Connect without specifying the database
conn = mysql.connector.connect(
    user=username,
    password=password,
    host=host
)

cursor = conn.cursor()
cursor.execute("SHOW DATABASES;")

for db in cursor:
    print(db[0])

cursor.close()
conn.close()

information_schema
mysql
performance_schema
sakila
sys
world


In [14]:
import mysql.connector

conn = mysql.connector.connect(
    user=username,
    password=password,
    host=host
)

cursor = conn.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS pgdsms;")
print("Database 'pgdsms' created successfully.")

cursor.close()
conn.close()

Database 'pgdsms' created successfully.


In [15]:
conn = mysql.connector.connect(
    user=username,
    password=password,
    host=host,
    database="pgdsms"
)

print("Connected successfully to pgdsms!")
conn.close()

Connected successfully to pgdsms!


In [19]:
import mysql.connector

# 1. Establish the connection to your newly created database
conn = mysql.connector.connect(
    user=username,
    password=password,
    host=host,
    database="pgdsms"
)

# 2. Create the cursor
cursor = conn.cursor()

# 3. Test running a query
cursor.execute("SHOW TABLES;")
print("Tables in pgdsms:")
for table in cursor:
    print(table[0])


Tables in pgdsms:


In [20]:
cursor = conn.cursor()
cursor

In [21]:
# Step 2: Show all tables
cursor.execute("SHOW TABLES;")
tables = cursor.fetchall()
print("Tables:", tables)

Tables: []


In [1]:
-- -------------------------------------------------------------
-- LMS Database Schema & Seed Data
-- -------------------------------------------------------------
CREATE DATABASE IF NOT EXISTS pgdsms;
USE pgdsms;

-- Clean slate
DROP TABLE IF EXISTS grades;
DROP TABLE IF EXISTS enrollments;
DROP TABLE IF EXISTS courses;
DROP TABLE IF EXISTS instructors;
DROP TABLE IF EXISTS students;

-- 1. Students Table
CREATE TABLE students (
    student_id INT AUTO_INCREMENT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) NOT NULL UNIQUE,
    phone VARCHAR(20) NOT NULL,
    enrollment_date DATE NOT NULL,
    status ENUM('Active', 'Inactive', 'Suspended') DEFAULT 'Active'
) ENGINE=InnoDB;

-- 2. Instructors Table
CREATE TABLE instructors (
    instructor_id INT AUTO_INCREMENT PRIMARY KEY,
    first_name VARCHAR(50) NOT NULL,
    last_name VARCHAR(50) NOT NULL,
    email VARCHAR(100) NOT NULL UNIQUE,
    specialization VARCHAR(100) NOT NULL,
    status ENUM('Active', 'On Leave', 'Inactive') DEFAULT 'Active'
) ENGINE=InnoDB;

-- 3. Courses Table
CREATE TABLE courses (
    course_id INT AUTO_INCREMENT PRIMARY KEY,
    course_name VARCHAR(120) NOT NULL,
    description TEXT,
    instructor_id INT,
    duration_weeks INT NOT NULL CHECK (duration_weeks > 0),
    fee DECIMAL(10, 2) NOT NULL CHECK (fee >= 0),
    status ENUM('Active', 'Upcoming', 'Archived') DEFAULT 'Active',
    CONSTRAINT fk_courses_instructor 
        FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id) 
        ON DELETE SET NULL 
        ON UPDATE CASCADE
) ENGINE=InnoDB;

-- 4. Enrollments Table (Many-to-Many Bridge with Unique Constraint)
CREATE TABLE enrollments (
    enrollment_id INT AUTO_INCREMENT PRIMARY KEY,
    student_id INT NOT NULL,
    course_id INT NOT NULL,
    enrollment_date DATE NOT NULL,
    status ENUM('Enrolled', 'Completed', 'Dropped') DEFAULT 'Enrolled',
    CONSTRAINT uq_student_course UNIQUE (student_id, course_id),
    CONSTRAINT fk_enrollments_student 
        FOREIGN KEY (student_id) REFERENCES students(student_id) 
        ON DELETE CASCADE 
        ON UPDATE CASCADE,
    CONSTRAINT fk_enrollments_course 
        FOREIGN KEY (course_id) REFERENCES courses(course_id) 
        ON DELETE CASCADE 
        ON UPDATE CASCADE
) ENGINE=InnoDB;

-- 5. Grades Table
CREATE TABLE grades (
    grade_id INT AUTO_INCREMENT PRIMARY KEY,
    enrollment_id INT NOT NULL UNIQUE,
    assignment_marks DECIMAL(5, 2) DEFAULT 0.00 CHECK (assignment_marks BETWEEN 0 AND 100),
    quiz_marks DECIMAL(5, 2) DEFAULT 0.00 CHECK (quiz_marks BETWEEN 0 AND 100),
    final_exam_marks DECIMAL(5, 2) DEFAULT 0.00 CHECK (final_exam_marks BETWEEN 0 AND 100),
    CONSTRAINT fk_grades_enrollment 
        FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id) 
        ON DELETE CASCADE 
        ON UPDATE CASCADE
) ENGINE=InnoDB;

-- -------------------------------------------------------------
-- Seed Data
-- -------------------------------------------------------------
INSERT INTO students (first_name, last_name, email, phone, enrollment_date, status) VALUES
('Ahmad', 'Raza', 'ahmad.raza@example.com', '03001234567', '2025-01-10', 'Active'),
('Sara', 'Khan', 'sara.khan@example.com', '03019876543', '2025-01-15', 'Active'),
('Bilal', 'Tariq', 'bilal.tariq@example.com', '03214567890', '2025-02-01', 'Inactive'),
('Zainab', 'Malik', 'zainab.malik@example.com', '03331122334', '2025-02-15', 'Active'),
('Hamza', 'Ali', 'hamza.ali@example.com', '03459988776', '2025-03-01', 'Active');

INSERT INTO instructors (first_name, last_name, email, specialization, status) VALUES
('Dr. Usman', 'Ghani', 'usman.ghani@example.com', 'Database Systems & Data Mining', 'Active'),
('Ayesha', 'Siddiqui', 'ayesha.s@example.com', 'Web Technologies & Cloud', 'Active'),
('Farhan', 'Akhtar', 'farhan.a@example.com', 'Artificial Intelligence', 'On Leave'),
('Naveed', 'Iqbal', 'naveed.i@example.com', 'Cybersecurity', 'Active');

INSERT INTO courses (course_name, description, instructor_id, duration_weeks, fee, status) VALUES
('Relational Databases & SQL', 'Comprehensive guide to MySQL, indexing, and joins.', 1, 10, 25000.00, 'Active'),
('Full-Stack Web with Python', 'Streamlit, Flask, and REST API integration.', 2, 12, 30000.00, 'Active'),
('Applied Machine Learning', 'Supervised and unsupervised ML models.', 3, 16, 45000.00, 'Upcoming'),
('Network Security Fundamentals', 'Firewalls, cryptography, and defense strategies.', 4, 8, 20000.00, 'Active'),
('Advanced Data Structures', 'Algorithmic efficiency and storage models.', 1, 8, 22000.00, 'Active');

INSERT INTO enrollments (student_id, course_id, enrollment_date, status) VALUES
(1, 1, '2025-01-12', 'Completed'),
(1, 2, '2025-01-18', 'Enrolled'),
(2, 1, '2025-01-20', 'Enrolled'),
(2, 4, '2025-02-05', 'Enrolled'),
(3, 1, '2025-02-03', 'Dropped'),
(4, 2, '2025-02-16', 'Completed'),
(5, 4, '2025-03-05', 'Enrolled');

INSERT INTO grades (enrollment_id, assignment_marks, quiz_marks, final_exam_marks) VALUES
(1, 92.00, 88.50, 94.00),
(2, 85.00, 78.00, 82.50),
(3, 76.50, 80.00, 85.00),
(4, 89.00, 91.00, 90.00),
(6, 95.00, 93.00, 98.00),
(7, 65.00, 70.00, 72.00);

SyntaxError: invalid syntax (977221188.py, line 1)